# 🚀 GPT

In this notebook, we'll walk through the steps required to train your own GPT model on the recipe review dataset

The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.  
https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/09_transformer/gpt/gpt.ipynb

https://www.kaggle.com/datasets/hugodarwood/epirecipes

The training runs an order of magnitude faster on colab (with TPU enabled) than on a laptop.

In [1]:
#%load_ext autoreload
#%autoreload 2
import numpy as np
import json
import re
import string
from IPython.display import display, HTML
#!pip install kaggle
import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks
from tensorflow import keras

import os

## 0. Parameters <a name="parameters"></a>

In [2]:
VOCAB_SIZE = 10000
MAX_LEN = 128          
EMBEDDING_DIM = 128   
KEY_DIM = 128          
N_HEADS = 4            
FEED_FORWARD_DIM = 512 
VALIDATION_SPLIT = 0.2 
SEED = 57
LOAD_MODEL = False
BATCH_SIZE = 32        
EPOCHS = 5

## 1. Load the data <a name="load"></a>

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hugodarwood/epirecipes")

print("Path to dataset files:", path)

c:\Users\Simon Seo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Simon Seo\.cache\kagglehub\datasets\hugodarwood\epirecipes\versions\2


In [4]:
# Find JSON files in the directory
json_files = [f for f in os.listdir(path) if f.endswith('.json')]

if json_files:
    # Load the first JSON file (or specify which one you want)
    json_file_path = os.path.join(path, json_files[0])
    print(f"\nLoading: {json_files[0]}")
    
    with open(json_file_path, 'r', encoding='utf-8') as json_data:
        recipe = json.load(json_data)
    
    print(f"Successfully loaded data with {len(recipe)} entries")
else:
    print("\nNo JSON files found. The dataset might be in another format (CSV, etc.)")


Loading: full_format_recipes.json
Successfully loaded data with 20130 entries


In [5]:
recipe[10]

{'directions': ['Heat oil in heavy large skillet over medium-high heat. Add shallots and minced rosemary and sauté until tender, about 3 minutes. Add yams and broth to skillet and bring to boil. Cover skillet, reduce heat to medium-low and simmer until yams are almost tender, about 10 minutes. Add cream and sprinkle lightly with nutmeg. Simmer uncovered until yams are very tender and liquid thickens and coats yams, about 4 minutes. Season with salt and pepper. (Can be made 1 day ahead. Transfer to microwave-safe dish. Chill until cold, then cover and keep chilled. Rewarm, covered, in microwave on medium-low heat.)'],
 'fat': 5.0,
 'date': '2004-08-20T04:00:00.000Z',
 'categories': ['Milk/Cream',
  'Dairy',
  'Side',
  'Thanksgiving',
  'Rosemary',
  'Sweet Potato/Yam',
  'Fall',
  'Nutmeg',
  'Simmer',
  'Bon Appétit'],
 'calories': 256.0,
 'desc': 'Simmering the yams fills them with flavor and yields a lovely coating.',
 'protein': 4.0,
 'rating': 3.75,
 'title': 'Yams Braised with Cr

In [6]:
filtered_data = [
    f"Recipe: {x['title']} | Category: {x['categories'][0]} | Rating: {x['rating']} | Description: {x['desc']} | Main Ingredient: {x['ingredients'][0]}"
    for x in recipe
    if isinstance(x, dict)
    and x.get('categories') and len(x['categories']) > 0
    and x.get('title')
    and x.get('ingredients') and len(x['ingredients']) > 0
    and x.get('desc')
]

In [7]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

13427 recipes loaded


In [8]:
example = filtered_data[25]
print(example)

Recipe: Deviled Ham  | Category: Food Processor | Rating: 3.125 | Description: Can be prepared in 45 minutes or less. | Main Ingredient: 3/4 pound cooked ham, cut into pieces


## 2. Tokenize the data <a name="tokenize"></a>

In [9]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s

text_data = [pad_punctuation(x) for x in filtered_data]

In [10]:
# Display an example of a recipe
example_data = text_data[25]
example_data

'Recipe : Deviled Ham | Category : Food Processor | Rating : 3 . 125 | Description : Can be prepared in 45 minutes or less . | Main Ingredient : 3 / 4 pound cooked ham , cut into pieces'

In [11]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [12]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [13]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [14]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: :
3: |
4: .
5: ,
6: and
7: the
8: recipe
9: -


In [15]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[   8    2 2433  361    3   12    2  153  318    3   14    2   23    4
  105    3   13    2   44   42   92   22  101   82   32   93    4    3
   10   11    2   23   21   19   81  197  361    5   80   61  194    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0]


## 3. Create the Training Set <a name="create"></a>

In [16]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y

train_ds = text_ds.map(prepare_inputs)

In [17]:
example_input_output = train_ds.take(1).get_single_element()

In [18]:
# Example Input
example_input_output[0][0]

<tf.Tensor: shape=(128,), dtype=int64, numpy=
array([   8,    2,  125,   15,  743,    6,  978,    9,  129,  135,    3,
         12,    2,  125,    3,   14,    2,   19,    4,   27,    3,   13,
          2,   25,  907,    9,  226, 3267,   18,  125, 1155,   30,  149,
        155, 1048,    4,    3,   10,   11,    2,  160,   88,   53,    9,
        379,  978,    9,  129,  135,    5,  173,  627,    5,  528,  109,
         28,   17,  213,   53, 1144,   29,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0])>

In [19]:
# Example Output (shifted by one token)
example_input_output[1][0]

<tf.Tensor: shape=(128,), dtype=int64, numpy=
array([   2,  125,   15,  743,    6,  978,    9,  129,  135,    3,   12,
          2,  125,    3,   14,    2,   19,    4,   27,    3,   13,    2,
         25,  907,    9,  226, 3267,   18,  125, 1155,   30,  149,  155,
       1048,    4,    3,   10,   11,    2,  160,   88,   53,    9,  379,
        978,    9,  129,  135,    5,  173,  627,    5,  528,  109,   28,
         17,  213,   53, 1144,   29,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0])>

## 5. Create the causal attention mask function <a name="causal"></a>

In [20]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)

np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [21]:
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [22]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [23]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss=[losses.SparseCategoricalCrossentropy(), None]
)

In [24]:
gpt.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 128)      │     1,296,384 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ [(None, None, 128),    │       396,032 │
│ (TransformerBlock)              │ (None, 4, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,982,416 (11.38 MB)

 Trainable params: 2,982,416 (11.38 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    gpt = models.load_model("./models/gpt", compile=True)

## 9. Train the Transformer <a name="train"></a>

In [26]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("Recipe : title", max_tokens=50, temperature=.7)

In [27]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5", # ckpt",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [28]:
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - loss: 6.6977
generated text:
Recipe : title 

420/420 ━━━━━━━━━━━━━━━━━━━━ 52s 119ms/step - loss: 5.1387
Epoch 2/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - loss: 2.7816
generated text:
Recipe : title and salad and garlic | category : reduce 75 | rating : 3 . 375 

420/420 ━━━━━━━━━━━━━━━━━━━━ 52s 125ms/step - loss: 2.6080
Epoch 3/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 2.2591
generated text:
Recipe : title - style soup | category : condiment / cream | rating : 3 . 75 | description : 6 ; you ' s from the thing to the roulades , and a fresh pie it for if you can be prepared in the perfect believe a little

420/420 ━━━━━━━━━━━━━━━━━━━━ 54s 129ms/step - loss: 2.1923
Epoch 4/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - loss: 2.0435
generated text:
Recipe : title | category : condiment / cream | rating : 3 . 75 | description : use this recipe is a almonds , the cake . | main ingredient : 1 1 large vegetable oil 

420/4

In [29]:
# Save the final model
#gpt.save("./models/gpt")
gpt.save("./gpt.keras")

# 3. Generate text using the Transformer

In [30]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [31]:
max_tok = MAX_LEN
info = text_generator.generate(
    "Recipe: Rating: 3.5", max_tokens=max_tok, temperature=1.0
)


generated text:
Recipe: Rating: 3.5 a crust cream with vegetable | orange salsicce dip | category : citrus | rating : 4 . 75 | description : " tails served on [UNK] involved of a meal . | main ingredient : 1 pound boning 



In [32]:
j=0.2
while(j<=1):
    for i in range(2):
        print(f"Temperature: {j}")
        info = text_generator.generate(    
            "Recipe: spicy macaroni with cheddar cheese:", max_tokens=max_tok, temperature=j
        )
    j += 0.2

Temperature: 0.2

generated text:
Recipe: spicy macaroni with cheddar cheese: | category : chicken | rating : 3 . 0 | description : this recipe is a great with a little [UNK] , and a day ahead . | main ingredient : 1 / 4 - inch - inch - inch pieces 

Temperature: 0.2

generated text:
Recipe: spicy macaroni with cheddar cheese: | category : salad | rating : 4 . 375 | description : this recipe is a great with a little [UNK] , and a classic . | main ingredient : 1 / 2 cups all - inch - inch - inch - inch - inch - inch - inch pieces 

Temperature: 0.4

generated text:
Recipe: spicy macaroni with cheddar cheese: | category : salad | rating : 3 . 375 | description : this recipe can be prepared in 45 minutes or less . | main ingredient : 2 cup water 

Temperature: 0.4

generated text:
Recipe: spicy macaroni with cheddar cheese: | category : soup / cream | rating : 4 . 375 | description : this recipe can be prepared in 45 minutes or less . | main ingredient : 1 / 2 cups all purpose flour 

Tem

In [33]:
title = ['Recipe: tomatoe sauce:', 'Buttered steak:', 'American breakfast:', 'Asian cuisine:']

j = 0.2
while(j<=1):
    for i in range(2):
        print(f"Temperature: {j}")
        info = text_generator.generate(    
            title[i], max_tokens=max_tok, temperature=j
        )
        info = text_generator.generate(    
            title[i+1], max_tokens=max_tok, temperature=j
        )
    j += 0.1

Temperature: 0.2

generated text:
Recipe: tomatoe sauce: | category : salad | rating : 4 . 375 | description : this recipe is a [UNK] , and a great with a great with a little [UNK] , and a little [UNK] , and [UNK] , and a nice with a classic . | main ingredient : 1 / 2 cup sugar 


generated text:
Buttered steak: [UNK] | category : soup / stew | rating : 4 . 0 | description : this recipe is a great with a great with a day ahead . | main ingredient : 1 / 2 cups ) 

Temperature: 0.2

generated text:
Buttered steak: | main ingredient : 1 / 2 


generated text:
American breakfast: | main ingredient : 1 / 2 ( 1 / 2 - inch - inch - 

Temperature: 0.30000000000000004

generated text:
Recipe: tomatoe sauce: | category : soup / cream | rating : 4 . 0 | description : this recipe is a classic , and the [UNK] of the [UNK] . | main ingredient : 1 / 4 cup sugar 


generated text:
Buttered steak: [UNK] [UNK] | category : cheese | rating : 4 . 375 | description : this recipe is a [UNK] [UNK] [UNK] of 

In [34]:
print_probs(info, vocab)

[UNK]:   	9.890000343322754%
|:   	9.100000381469727%
the:   	3.2799999713897705%
with:   	1.5%
and:   	1.0800000429153442%
--------



[UNK]:   	3.4700000286102295%
accompaniment:   	2.7899999618530273%
easy:   	0.9900000095367432%
and:   	0.9800000190734863%
-:   	0.8100000023841858%
--------



|:   	70.8499984741211%
with:   	17.43000030517578%
and:   	0.9599999785423279%
salad:   	0.550000011920929%
,:   	0.5199999809265137%
--------



lemon:   	2.240000009536743%
tomato:   	1.5800000429153442%
red:   	1.4600000381469727%
roasted:   	1.4500000476837158%
orange:   	1.2799999713897705%
--------



chicken:   	7.739999771118164%
with:   	2.819999933242798%
pork:   	2.809999942779541%
and:   	2.009999990463257%
beef:   	1.350000023841858%
--------



...

2:   	44.43000030517578%
4:   	40.29999923706055%
3:   	7.449999809265137%
1:   	1.0800000429153442%
5:   	0.7900000214576721%
--------



cups:   	17.59000015258789%
tablespoons:   	12.449999809265137%
cup:   	10.640000343322754%
-:   	10.149999618530273%
pounds:   	7.739999771118164%
--------



(:   	7.690000057220459%
sugar:   	5.820000171661377%
):   	4.940000057220459%
olive:   	4.619999885559082%
all:   	4.21999979019165%
--------



chicken:   	3.819999933242798%
beef:   	3.0199999809265137%
:   	2.4600000381469727%
pork:   	2.119999885559082%
fresh:   	1.4800000190734863%
--------



:   	32.5%
,:   	10.920000076293945%
with:   	4.369999885559082%
and:   	3.4800000190734863%
(:   	3.4000000953674316%
--------

